# Notebook 06 — Structured Outputs

Reliably extract JSON-structured data using pre-fill and tool_choice.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))
from utils.anthropic_client import client, FAST_MODEL


## Method 1: Pre-fill + stop sequence

In [ ]:
def extract_json_prefill(text, fields):
    field_list = ", ".join(f'"{f}"' for f in fields)
    r = client.messages.create(
        model=FAST_MODEL, max_tokens=256,
        stop_sequences=["}"],
        messages=[
            {"role": "user",      "content": f"Extract {field_list} as JSON from:\n\n{text}"},
            {"role": "assistant", "content": "{"},
        ],
    )
    return json.loads("{" + r.content[0].text + "}")

result = extract_json_prefill(
    "Sarah Connor is 35 years old and lives in Los Angeles.",
    ["name", "age", "city"]
)
print(result)


## Method 2: Forced tool use

In [ ]:
EXTRACT_TOOL = {
    "name": "extract_person",
    "description": "Extract structured person data.",
    "input_schema": {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "age":  {"type": "integer"},
            "city": {"type": "string"},
        },
        "required": ["name", "age", "city"],
    },
}

r = client.messages.create(
    model=FAST_MODEL, max_tokens=256,
    tools=[EXTRACT_TOOL], tool_choice={"type": "any"},
    messages=[{"role": "user", "content": "Extract from: John Doe, 28, from Seattle."}],
)
for block in r.content:
    if block.type == "tool_use":
        print(block.input)


## Exercise
Extract a product listing (name, price, category) from free-form text.

In [ ]:
# Your code here
